- Entradas: "PwmD", "PwmE", "sPwm", "dPwm"
- Saida: Theta
- Loss = L_d + L_p 


In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib
import os
TITLES = [
    "ZZxReto", # Train
    "ZZy1", # Train
    "ZZx2",  # Val
    "ZZy2", # Val
    "LSG-1", # Test
    "LSG-2", # Test
    "ZZx1-inv", # Test
    "ZZx1",  # Test
    "ZZx2-inv", # Test
    "semiCirc", # Test
]

PREDICTORS = ["e_a_d", "e_a_e", "Se_a", "De_a"]   
TARGET_INT = ["theta"]  
TARGET = ["dtheta"]
     
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   
     
TIME_STEPS = 9
TS = 0.07
PLOT = False

In [21]:
Datasets = []
for title in TITLES:
    df = pd.read_excel("./../../00-Data/SavgolDatasets.xlsx", sheet_name=title)
    Datasets.append(df)

In [22]:
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"d{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS
    
    Dataset["Se_a"] = Dataset["e_a_d"] + Dataset["e_a_e"]
    Dataset["De_a"] = Dataset["e_a_d"] - Dataset["e_a_e"]

    Dataset = Dataset.dropna(subset=[f"d{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [23]:

NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)

# concatena
Train = pd.concat([Train1, Train2], ignore_index=True)

for i in range(8):
    CurrentTestDataset = Datasets[i + 2].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

Val = pd.concat([NormDatasets[2], NormDatasets[3]], ignore_index=True)

In [24]:
os.makedirs("./scalers", exist_ok=True)
os.makedirs("./Data", exist_ok=True)

with pd.ExcelWriter("./Data/NormDatasets.xlsx", engine="openpyxl") as writer_norm:
    for title, normDataset in zip(TITLES, NormDatasets):
        normDataset.to_excel(writer_norm, sheet_name=title[:31], index=False)

with pd.ExcelWriter("./Data/Datasets.xlsx", engine="openpyxl") as writer:
    for title, Dataset in zip(TITLES, Datasets):
        Dataset.to_excel(writer, sheet_name=title[:31], index=False)
        
joblib.dump(SCALER, "./scalers/scaler.pkl")
joblib.dump(OUT_SCALER, "./scalers/out_scaler.pkl")

mean_tf = tf.constant(OUT_SCALER.mean_[0], dtype=tf.float32)
std_tf  = tf.constant(OUT_SCALER.scale_[0], dtype=tf.float32)        

In [25]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences(Val[PREDICTORS], Val[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(y_val)}")

Dimensão da entrada: (2655, 9, 4)
Dimensão da saida: (2655, 1)
Dimensão da entrada: (3021, 9, 4)
Dimensão da saida: (3021, 1)


In [26]:
phi_d_train =  Train["phi_d"].values[:len(x_train)]
phi_e_train =  Train["phi_e"].values[:len(x_train)]
theta_train = Train["theta"].values[:len(x_train)]

print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada fisica : {np.shape(phi_d_train)}")
print(f"Dimensão da entrada fisica: {np.shape(phi_e_train)}")

Dimensão da entrada: (2655, 9, 4)
Dimensão da saida: (2655, 1)
Dimensão da entrada fisica : (2655,)
Dimensão da entrada fisica: (2655,)


$$ \dot{\theta} = \frac{R}{2L} (\phi_d - \phi_e) $$
$$ \dot{x} = \frac{R}{2} (\phi_d + \phi_e) (\cos(\theta))$$
$$ \dot{y} = \frac{R}{2} (\phi_d + \phi_e) (\sin(\theta)) $$

In [27]:
R = tf.constant(0.0328, dtype=tf.float32)
L = tf.constant(0.0615, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def CinematicModel(phi_d, phi_e, theta):

    dtheta_cin = (R / (2 * L)) * (phi_d - phi_e)
    return [dtheta_cin]
    

In [28]:

def NumericalIntegration(dataset, dq):

    q = [None] * OUTPUT_SIZE

    init_vals = np.array([
        dataset[name].iloc[0] for name in TARGET_INT
    ])

    for j in range(OUTPUT_SIZE):
        q[j] = init_vals[j] + np.cumsum(dq[j] * TS)

    return q

def GetCin(dataset): 
    dq = CinematicModel(tf.convert_to_tensor(dataset["phi_d"].values, dtype=tf.float32),
                        tf.convert_to_tensor(dataset["phi_e"].values, dtype=tf.float32), 
                        tf.convert_to_tensor(dataset["theta"].values, dtype=tf.float32))
    q = NumericalIntegration(dataset, dq)
    return np.vstack(dq).T, np.vstack(q).T

In [29]:
def BuildRNN(architecture, initializer, regularizer):

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

    for i, units in enumerate(architecture):

        return_sequences = (i < len(architecture) - 1)

        model.add(
            tf.keras.layers.SimpleRNN(
                units,
                activation='tanh',
                return_sequences=return_sequences,
                kernel_initializer=initializer,
                kernel_regularizer=regularizer,
                recurrent_regularizer=regularizer,
                bias_regularizer=regularizer
            )
        )

    model.add(
        tf.keras.layers.Dense(
            OUTPUT_SIZE,
            activation="linear",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer,
            bias_regularizer=regularizer
        )
    )

    return model

In [30]:
@tf.function
def train_step(model, optimizer, x, dy, phi_d, phi_e, theta0, Ld, Lp):
    weights = 1 + 3 * tf.abs(dy)

    with tf.GradientTape() as tape:

        dy_pred = model(x, training=True)
        
        # loss dos dados
        data_loss = tf.reduce_mean(weights * tf.square(dy_pred - dy))
        
        # termo físico
        physics = tf.stack(CinematicModel(phi_d, phi_e, theta0), axis=1)   
             
        # normalização correta
        physics_norm = (physics - mean_tf) / std_tf

        physics_loss = tf.reduce_mean(tf.square(dy_pred - physics_norm))

        loss = Ld * data_loss +  Lp * physics_loss 

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss, data_loss, physics_loss

In [31]:
phi_d_train = tf.convert_to_tensor(phi_d_train, dtype=tf.float32)
phi_e_train = tf.convert_to_tensor(phi_e_train, dtype=tf.float32)
theta_train = tf.convert_to_tensor(theta_train, dtype=tf.float32)

x_train_tf = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.float32)

x_val_tf = tf.convert_to_tensor(x_val, dtype=tf.float32)
y_val_tf = tf.convert_to_tensor(y_val, dtype=tf.float32)

In [32]:
def EarlyStopping(model, best_loss, counter, best_weights, min_delta=1e-3):
    val_pred = model(x_val_tf, training=False)
    val_loss = tf.reduce_mean(tf.square(val_pred - y_val_tf))
    
    if val_loss < (best_loss - min_delta):
        best_loss = val_loss
        counter = 0
        best_weights = model.get_weights()

    else:
        counter += 1

    return best_loss, counter, best_weights, val_loss

def TrainPINN(model, optimizer, Ld, Lp, patience=200, best_loss=np.inf):
    counter = 0
    best_weights = model.get_weights()

    for epoch in range(20000):

        loss, data_loss, physics_loss = train_step(model, optimizer, x_train_tf, y_train_tf,
                          phi_d_train, phi_e_train, theta_train, Ld, Lp)
        
        best_loss, counter, best_weights, val_loss =  EarlyStopping(model, best_loss, counter, best_weights)
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            model.set_weights(best_weights)
            break

        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Train Loss {loss.numpy():.6f} | data Loss {data_loss.numpy():.6f} |  physics Loss {physics_loss.numpy():.6f} | Val Loss {val_loss.numpy():.6f}")

In [33]:
def PlotOut(ax, title, target_name, y_true, y_pred, y_cin):
    time = (np.arange(len(y_pred)).astype(float) * 0.07).round(5)

    ax.plot(time, y_true, '-', linewidth=1.5, label='Amostras Reais')
    ax.plot(time, y_pred, '--', linewidth=1.5, label='Valores preditos')
    ax.plot(time, y_cin, ':', linewidth=2, label='Modelo Cinemático')

    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)
    ax.legend()
    ax.grid(True)


def EvalModel(model):
    from sklearn.metrics import r2_score, mean_squared_error

    n_targets = len(TARGET)
    n_datasets = len(Datasets)

    if PLOT:
        fig, axs = plt.subplots(
            n_datasets,
            2 * n_targets,
            figsize=(6 * 2 * n_targets, 4 * n_datasets)
        )
        axs = np.atleast_2d(axs)

    metrics = {name: {} for name in TARGET_INT}

    for i, NormDataset in enumerate(NormDatasets):

        x = NormDataset[PREDICTORS]
        y = Datasets[i][TARGET_INT]
        dy_true = Datasets[i][TARGET].values

        x, y = CreateSequences(x, y, TIME_STEPS)

        # alinhar derivada
        dy_true = dy_true[TIME_STEPS:]

        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()
        dy_pred = OUT_SCALER.inverse_transform(pred)

        y_true = y.copy()
        y_pred = np.zeros_like(dy_pred)

        dy_cin, y_cin = GetCin(Datasets[i])
        y_cin = y_cin[:y_true.shape[0]]
        dy_cin = dy_cin[:dy_pred.shape[0]]

        init_vals = np.array([Datasets[i][name].iloc[0] for name in TARGET_INT])

        for j in range(n_targets):
            y_pred[:, j] = init_vals[j] + np.cumsum(dy_pred[:, j] * TS)

        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = r2_score(dy_true[:, j], dy_pred[:, j])

            key_r2 = f"R2_{TITLES[i]}"
            key_mse = f"MSE_{TITLES[i]}"

            metrics[name].setdefault(key_r2, []).append(r2)
            metrics[name].setdefault(key_mse, []).append(mse)

            print(f"{name} | {TITLES[i]} -> R² = {r2:.4f}, R² diff = {mse:.4f}")

            if PLOT:
                ax_y = axs[i, j]
                PlotOut(ax_y, TITLES[i], name,
                        y_true[:, j], y_pred[:, j], y_cin[:, j])

                ax_dy = axs[i, j + n_targets]
                PlotOut(ax_dy, TITLES[i], f"d{name}",
                        dy_true[:, j], dy_pred[:, j], dy_cin[:, j])

    if PLOT:
        plt.tight_layout()

    return metrics

In [34]:
def to_scalar(x):
    return float(x[0]) if isinstance(x, list) else float(x)

In [35]:
def UpdateRow(metrics, arch, Ld, Lp, r, seed, excel_file):

    model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_Ld{Ld}_Lp{Lp}_seed{seed}"

    row = {
        "model": model_name,
        "Neurons": arch,
        "Ld": Ld,
        "Lp": Lp,
        "reg": r,
        "seed": seed,
    }

    for name in TARGET_INT:
        entry = {}
        for title in TITLES:
            safe_title = title.replace("-", "_")  
            entry[f"R2_{safe_title}_{name}"] = to_scalar(metrics[name][f"R2_{title}"])
            entry[f"MSE_{safe_title}_{name}"] = to_scalar(metrics[name][f"MSE_{title}"])
        row.update(entry)
        df = pd.DataFrame([row])

    try:
        old = pd.read_excel(excel_file)
        new_df = pd.concat([old, df], ignore_index=True)
        new_df.to_excel(excel_file, index=False)
    except FileNotFoundError:
        df.to_excel(excel_file, index=False)

    print(f"Modelo {arch} | Ld={Ld} Lp={Lp} r={r} seed={seed} salvo.")


In [36]:
def ExportModel(model, model_name):

    os.makedirs("weights", exist_ok=True)
    os.makedirs("models", exist_ok=True)

    weights_path = f"weights/{model_name}.weights.h5"
    model_path = f"models/{model_name}.keras"

    model.save_weights(weights_path)
    model.save(model_path)

    print(f"Modelo salvo em:\n{model_path}\n{weights_path}")

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import initializers
from itertools import product
import numpy as np
import tensorflow as tf

N_MODELS = 5

seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

# architectures = [[n] for n in range(81, 101)]

FirstLayers = [ [25],[28],[29],[32],
                [33],[35],[40],[45],
]

architectures = [
    [fl[0], sl]
    for fl in FirstLayers
    for sl in range(
        max(1, int(0.3 * fl[0])),
        int(0.7 * fl[0])
    )
]

# 🔹 retomar a partir do último modelo salvo
# last_saved = [25, 7]

# if last_saved in architectures:
#     idx = architectures.index(last_saved)
#     architectures = architectures[idx + 1:]  # começa DEPOIS do último salvo
#     print(f"Retomando a partir de {last_saved} (índice {idx}). Restam {len(architectures)} arquiteturas.")
# else:
#     print(f"⚠️ {last_saved} não encontrado na lista — rodando tudo do zero.")

print(len(architectures))
print(architectures)

Ld_Lp = [[0.3, 0.7], [0.7, 0.3]]
r_values = [0.01, 0.9]

results = {}

# produto cartesiano de todos hiperparâmetros
for arch, (Ld, Lp), r in product(architectures, Ld_Lp, r_values):

    for i, s in enumerate(seeds):

        tf.keras.backend.clear_session()

        init = initializers.RandomNormal(seed=int(s))
        reg = tf.keras.regularizers.l2(r)
        model = BuildRNN(arch, init, reg)
        model.build((None, TIME_STEPS, INPUT_SIZE))

        opt = Adam(learning_rate=0.001)
        opt.build(model.trainable_variables)

        TrainPINN(
            model,
            Ld=Ld,
            Lp=Lp,
            optimizer=opt
        )
        model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_Ld{Ld}_Lp{Lp}_seed{s}"
        ExportModel(model, model_name=model_name)
        metrics = EvalModel(model)
        UpdateRow(metrics, arch, Ld, Lp, r, s, excel_file="resultados-2l.xlsx")

108
[[25, 7], [25, 8], [25, 9], [25, 10], [25, 11], [25, 12], [25, 13], [25, 14], [25, 15], [25, 16], [28, 8], [28, 9], [28, 10], [28, 11], [28, 12], [28, 13], [28, 14], [28, 15], [28, 16], [28, 17], [28, 18], [29, 8], [29, 9], [29, 10], [29, 11], [29, 12], [29, 13], [29, 14], [29, 15], [29, 16], [29, 17], [29, 18], [29, 19], [32, 9], [32, 10], [32, 11], [32, 12], [32, 13], [32, 14], [32, 15], [32, 16], [32, 17], [32, 18], [32, 19], [32, 20], [32, 21], [33, 9], [33, 10], [33, 11], [33, 12], [33, 13], [33, 14], [33, 15], [33, 16], [33, 17], [33, 18], [33, 19], [33, 20], [33, 21], [33, 22], [35, 10], [35, 11], [35, 12], [35, 13], [35, 14], [35, 15], [35, 16], [35, 17], [35, 18], [35, 19], [35, 20], [35, 21], [35, 22], [35, 23], [40, 12], [40, 13], [40, 14], [40, 15], [40, 16], [40, 17], [40, 18], [40, 19], [40, 20], [40, 21], [40, 22], [40, 23], [40, 24], [40, 25], [40, 26], [40, 27], [45, 13], [45, 14], [45, 15], [45, 16], [45, 17], [45, 18], [45, 19], [45, 20], [45, 21], [45, 22], [45,